### **PASO 1: Instalación de Paquetes Necesarios**
Se instalan las bibliotecas necesarias para que el chatbot funcione correctamente.

- **Configuración del Entorno:** Se desactiva la paralelización de los tokenizadores, lo cual puede generar advertencias innecesarias.
- **Verificación de la Versión de Python:** La función `verificar_version_python` verifica si la versión de Python es la requerida (3.10.12). Si no es la misma, se muestra una advertencia para que el usuario sepa que la versión no coincide con la recomendada.
- **Instalación de Paquetes:** Se instalan varias bibliotecas necesarias para el chatbot, utilizando pip en el entorno actual. Esto incluye bibliotecas para:
  - Transformadores (transformers)
  - Embeddings y modelos de texto (sentence_transformers)
  - Búsqueda y indexación HNSW (hnswlib)
  - Manejo de PDFs (PyPDF2)
  - Variables de entorno (python-dotenv)
  - Gestión de reintentos en APIs (tenacity)
  - Integración con Gemini (llama-index y otros paquetes relacionados)
  - Barra de progreso (tqdm)

In [ ]:
import os
import sys

# Desactiva la advertencia configurando la variable a "false"
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Evita advertencias de paralelización de tokenizadores

# Verificar versión de Python
def verificar_version_python():
    """
    Verifica si la versión de Python instalada es la versión recomendada.
    """
    # Versión requerida: 3.10.12
    version_requerida = (3, 10, 12)  # Versión de Python recomendada
    version_actual = sys.version_info[:3]  # Obtiene (major, minor, micro)

    if version_actual == version_requerida:
        print(f"✅ Versión de Python {'.'.join(map(str, version_actual))} detectada. ¡Éxito!")
    else:
        version_actual_str = '.'.join(map(str, version_actual))
        version_requerida_str = '.'.join(map(str, version_requerida))
        print(f"⚠️ Advertencia: Se detectó Python {version_actual_str}. Se recomienda usar Python {version_requerida_str} al instalar librerías.")

verificar_version_python()  # Llama a la función de verificación de versión

# Instalación de bibliotecas necesarias
%pip install transformers  # Instala la librería transformers para modelos LLM
%pip install sentence_transformers  # Instala sentence-transformers para crear embeddings
%pip install hnswlib  # Instala hnswlib para búsquedas rápidas de similitud
%pip install numpy<2.0  # Especificar versión de numpy menor que 2.0
%pip install PyPDF2  # Para extraer texto de archivos PDF
%pip install python-dotenv  # Para manejo de variables de entorno desde .env
%pip install tenacity  # Para manejar reintentos en llamadas API
%pip install llama-index  # Para integración con Gemini
%pip install llama-index-llms-gemini  # Para usar Gemini como LLM
%pip install llama-index-embeddings-gemini  # Para embeddings relacionados con Gemini
%pip install tqdm  # Mostrar barras de progreso durante la ejecución de tareas

### **PASO 2: Importar Librerías y Configurar Logging**
Se importan las librerías necesarias y se configura un sistema de logs para monitorear todo el flujo.

- **Importación de Librerías:** Se importan las librerías necesarias para el chatbot, incluyendo herramientas de procesamiento de texto, embeddings, manejo de PDFs y la integración con Gemini.
- **Configuración de Logging:** Se configura el sistema de logs para imprimir mensajes informativos en consola, facilitando la depuración del flujo del código.
- **Carga de Variables de Entorno:** Se carga un archivo `.env` con variables de entorno necesarias para el correcto funcionamiento del chatbot.

In [ ]:
# Importar todas las librerías necesarias
import os  # Para interactuar con el sistema operativo, como crear directorios o manejar rutas
import json  # Para manipulación de archivos JSON
import logging  # Para gestionar registros y logueos
import hnswlib  # Librería para trabajar con índices HNSW (para búsqueda rápida de embeddings)
from transformers import AutoTokenizer, AutoModelForCausalLM  # Librerías de Hugging Face para modelos de lenguaje
from sentence_transformers import SentenceTransformer, util  # Para generar embeddings y usar similitud de oraciones
import numpy as np  # Para manejo de arrays y cálculos matemáticos
from dotenv import load_dotenv  # Para cargar variables de entorno desde un archivo .env
from PyPDF2 import PdfReader  # Para leer y extraer texto de archivos PDF
from tenacity import retry, wait_exponential, stop_after_attempt  # Para manejo de reintentos con exponencial backoff
from llama_index.llms.gemini import Gemini  # Para trabajar con el modelo Gemini en la generación de respuestas
from llama_index.core.llms import ChatMessage  # Para el manejo de mensajes de chat con el modelo Gemini
import time  # Para medir tiempos y realizar pausas
import hashlib  # Para crear hashes, usado en el caché de respuestas
import random  # Para elegir respuestas aleatorias

# Configuración de logging para imprimir todo en consola
logging.basicConfig(
    level=logging.INFO,  # Establece el nivel de log para que se muestren mensajes informativos y más graves
    format='%(asctime)s - %(levelname)s - %(message)s',  # Formato para los logs
    handlers=[logging.StreamHandler()]  # Solo consola (por defecto, sin archivos de log)
)

# Mensaje de log para confirmar que las librerías se han importado correctamente
logging.info("Librerías importadas correctamente.")  

# Cargar variables de entorno desde un archivo .env
# Esto permite cargar configuraciones sensibles o específicas para el entorno desde un archivo
load_dotenv()

# Mensaje de log para confirmar que las variables de entorno se han cargado
logging.info("Variables de entorno cargadas desde el archivo .env.")


### **PASO 3: Cargar Documentos**
Este paso carga documentos desde un archivo o un directorio. Soporta formatos .txt, .json y .pdf.


- **Carga de Documentos desde Archivo o Directorio:** La función `load_documents` carga documentos desde un archivo o un directorio. Soporta los formatos `.txt`, `.json`, y `.pdf`.
  - Si se carga desde un directorio, se recorren todos los archivos con las extensiones correspondientes y se extrae el contenido de cada uno.
  - Si se carga desde un archivo individual, simplemente se extrae el contenido del archivo.
  
- **Extracción de Contenido:** Dependiendo del tipo de archivo (txt, json, pdf):
  - **.txt:** El contenido se divide en unidades usando el delimitador `\n-----\n`.
  - **.json:** El archivo se lee y se carga como un objeto JSON.
  - **.pdf:** El texto se extrae de las páginas del PDF.

- **Manejo de Errores:** Si ocurre un error durante la carga o extracción del contenido, se registra un error en los logs.

- **Resultado Final:** Devuelve una lista de documentos cargados con el nombre de archivo y su contenido.

- **Ejemplo de uso:** Se carga un directorio con documentos (en este caso 'data') y se registra cuántos documentos fueron cargados correctamente.

In [ ]:
def load_documents(source, is_directory=False):
    """
    Carga documentos desde un archivo o un directorio. Soporta .txt, .json y .pdf.
    Además, divide los archivos .txt en unidades usando un delimitador específico.
    """
    loaded_files = []  # Lista para almacenar los documentos cargados

    # Verificar si la ruta existe
    if not os.path.exists(source):
        logging.error(f"La fuente '{source}' no existe.")  # Loguear error si la fuente no existe
        raise FileNotFoundError(f"La fuente '{source}' no se encontró.")  # Lanzar excepción

    if is_directory:
        # Si es un directorio, cargar todos los archivos .txt, .json, .pdf dentro de él
        logging.info(f"Iniciando carga desde el directorio: {source}.")
        for filename in os.listdir(source):
            filepath = os.path.join(source, filename)
            if os.path.isfile(filepath) and filepath.endswith(('.txt', '.json', '.pdf')):
                # Extraer contenido de cada archivo compatible
                content = extract_content(filepath)
                if content:
                    # Agregar contenido a la lista de archivos cargados
                    loaded_files.append({"filename": filename, "content": content})
                    logging.info(f"Archivo '{filename}' cargado correctamente.")
    else:
        # Si no es un directorio, cargar el archivo específico
        logging.info(f"Iniciando carga del archivo: {source}.")
        content = extract_content(source)
        if content:
            # Agregar contenido del archivo a la lista de archivos cargados
            loaded_files.append({"filename": os.path.basename(source), "content": content})
            logging.info(f"Archivo '{os.path.basename(source)}' cargado correctamente.")

    # Loguear la cantidad total de documentos cargados
    logging.info(f"{len(loaded_files)} documentos cargados.")
    return loaded_files  # Devolver la lista de documentos cargados

def extract_content(filepath):
    """
    Extrae el contenido del archivo según su tipo (.txt, .json, .pdf).
    Si el archivo es .txt, lo divide en unidades por el delimitador '\n------\n'.
    """
    try:
        # Verificar si el archivo es .txt
        if filepath.endswith('.txt'):
            with open(filepath, 'r', encoding='utf-8') as file:
                content = file.read()  # Leer todo el contenido del archivo
            # Dividir el contenido en unidades por el delimitador '\n-----\n'
            units = content.split("\n-----\n")
            return units  # Devolver una lista de unidades separadas
        # Verificar si el archivo es .json
        elif filepath.endswith('.json'):
            with open(filepath, 'r', encoding='utf-8') as file:
                return json.load(file)  # Devolver el contenido del archivo JSON
        # Verificar si el archivo es .pdf
        elif filepath.endswith('.pdf'):
            reader = PdfReader(filepath)  # Leer el archivo PDF
            return ''.join(page.extract_text() or '' for page in reader.pages)  # Extraer texto de cada página y unirlo
    except Exception as e:
        logging.error(f"Error al extraer contenido de '{filepath}': {e}")  # Loguear error si algo falla
        return None  # Devolver None si hay un error

# Cargar documentos (True: busca en directorio, False: busca archivo)
ruta_fuente = 'data'  # Usando un directorio como ejemplo
documentos = load_documents(ruta_fuente, is_directory=True)  # Llamada a la función para cargar los documentos

# Visualizar cuántos documentos fueron cargados
logging.info(f"Se cargaron {len(documentos)} documentos exitosamente.")  # Loguear cuántos documentos se cargaron


### **PASO 4: Configurar la Clave API de Gemini**
Configura la conexión con el modelo de lenguaje Gemini usando la clave API.

- **Configuración de la Conexión con Gemini:** En este paso, se configura la instancia del modelo de lenguaje Gemini utilizando una clave API proporcionada a través de las variables de entorno.
  - Se obtiene la clave API desde el archivo `.env` usando `os.getenv("GEMINI_API_KEY")`.
  - Si no se encuentra la clave API, se lanza un error informando que la configuración no está completa y se recomienda añadir la clave al archivo `.env`.
  - Si la clave está presente, se inicializa la instancia de Gemini con esa clave.

- **Manejo de Errores:** Si no se encuentra la clave API, se registra un error y se detiene el proceso con una excepción.
  
- **Resultado Final:** Una vez configurado correctamente, se registra en los logs que Gemini ha sido configurado con éxito.

- **Ejemplo de uso:** La función `configure_gemini()` es llamada para inicializar la conexión con el modelo Gemini.

In [ ]:
gemini_llm = None  # Inicializa la variable global para la instancia de Gemini

def configure_gemini():
    """
    Configura la instancia de Gemini usando la clave API almacenada en las variables de entorno.
    Si no se encuentra la clave, se lanza un error.
    """
    global gemini_llm  # Indica que estamos modificando la variable global 'gemini_llm'
    
    # Obtener la clave API de Gemini desde las variables de entorno
    api_key = os.getenv("GEMINI_API_KEY")
    
    # Verificar si la clave API está disponible
    if not api_key:
        logging.error("La clave API de Gemini no está configurada.")  # Loguear error si no se encuentra la clave
        raise EnvironmentError("Configura GEMINI_API_KEY en tu archivo .env.")  # Lanzar excepción si no se encuentra la clave API
    
    # Configurar la instancia de Gemini con la clave API obtenida
    gemini_llm = Gemini(api_key=api_key)
    
    # Loguear que la configuración se ha realizado correctamente
    logging.info("Gemini configurado correctamente.")

# Configurar Gemini al ejecutar el código
configure_gemini()  # Llamada a la función para configurar Gemini


### **PASO 5: Configurar el Modelo de Embeddings**
Configurar el modelo de embeddings.

- **Selección del Modelo de Embeddings:** En este paso, se selecciona y carga un modelo preentrenado para generar embeddings. Se utiliza el modelo `paraphrase-multilingual-MiniLM-L12-v2`, el cual es ideal para trabajar con múltiples idiomas y generar representaciones vectoriales de texto.

- **Función `doc_enfermedad`:** 
  - **Objetivo:** Esta función toma una pregunta y busca el documento que más se asemeje a ella mediante comparación de embeddings.
  - **Proceso:**
    - Primero, la pregunta se codifica en un embedding utilizando el modelo de embeddings cargado.
    - Luego, se obtiene el nombre de cada archivo cargado en el paso anterior (`documentos`).
    - Los nombres de los archivos se convierten también en embeddings utilizando el mismo modelo.
    - Se calcula la similitud coseno entre el embedding de la pregunta y los embeddings de los archivos.
    - Se selecciona el índice del archivo con la mayor similitud para identificar el documento que más corresponde con la pregunta.

- **Resultado:** La función devuelve el índice del archivo que tiene el mayor nivel de similitud con la pregunta.


In [ ]:
# Modelo multilingüe de Sentence-Transformers para generar embeddings de preguntas y documentos
model_name = "paraphrase-multilingual-MiniLM-L12-v2"  # Este modelo está diseñado para trabajar con múltiples idiomas
model = SentenceTransformer(model_name)  # Cargar el modelo de Sentence-Transformers

def doc_enfermedad(pregunta):
    """
    Identifica el archivo que contiene la información relevante sobre la enfermedad mencionada en la pregunta.
    Utiliza los embeddings generados por el modelo multilingüe para encontrar el archivo más relevante comparando
    la pregunta con los nombres de los archivos de los documentos cargados.
    """
    # Generar el embedding de la pregunta utilizando el modelo
    preg_embedding = model.encode(pregunta)
    
    # Obtener los nombres de los archivos cargados en 'documentos'
    archivos = [documentos[i]['filename'] for i in range(len(documentos))]
    
    # Generar embeddings de los nombres de los archivos utilizando el mismo modelo
    doc_filenames_embeddings = [model.encode(name) for name in archivos]
    
    # Calcular la similitud de coseno entre el embedding de la pregunta y cada uno de los embeddings de los archivos
    similarities = [util.cos_sim(preg_embedding, doc_emb).item() for doc_emb in doc_filenames_embeddings]
    
    # Identificar el índice del archivo con la mayor similitud con la pregunta
    max_index = similarities.index(max(similarities))
    
    return max_index  # Devuelve el índice del documento más relevante


### **PASO 6: Crear Clases para Documentos e Índices**
Crear clases para documentos e índices.

- **Clase `Document`:**
  - **Propósito:** Representa un documento con su contenido y metadatos.
  - **Atributos:**
    - `page_content`: El texto del documento.
    - `metadata`: Un diccionario opcional con información adicional sobre el documento (título, resumen, tipo de estudio, países, fases, ID en ClinicalTrial).
  - **Método `__str__`:** Devuelve una representación en formato de texto del documento, mostrando los metadatos del mismo de manera organizada.

- **Clase `HNSWIndex`:**
  - **Propósito:** Implementa un índice utilizando la librería `hnswlib` para la búsqueda de similitud basada en los embeddings de los documentos.
  - **Atributos:**
    - `dimension`: La dimensión de los embeddings (debe coincidir con la dimensión del modelo de embeddings).
    - `index`: El índice HNSW creado con la librería `hnswlib`.
    - `metadata`: Una lista con los metadatos correspondientes a los documentos.
  - **Métodos:**
    - `__init__`: Inicializa el índice HNSW con los embeddings y los parámetros como el espacio de similitud (por defecto 'cosine'), `ef_construction` y `M`.
    - `similarity_search`: Realiza una búsqueda de similitud utilizando el vector de consulta y devuelve los `k` documentos más similares, junto con las distancias a esos documentos.

- **Resultado:** Este paso crea las clases necesarias para gestionar los documentos y realizar búsquedas eficientes de similitudes entre los embeddings.


In [ ]:
# Clase para representar documentos
class Document:
    def __init__(self, text, metadata=None):
        """
        Inicializa un documento con contenido y metadatos opcionales.
        
        :param text: El contenido del documento.
        :param metadata: Los metadatos asociados al documento (por defecto vacío).
        """
        self.page_content = text  # Contenido del documento
        self.metadata = metadata or {}  # Metadatos del documento (diccionario vacío si no se proporciona)

    def __str__(self):
        """
        Devuelve una representación en formato string del documento, mostrando sus metadatos de forma segura.
        
        :return: String con los detalles del documento (título, resumen, tipo de estudio, países, fase, ID de estudio).
        """
        # Acceder a los metadatos de forma segura utilizando .get() para evitar errores si no existen
        return (
            f"Título: {self.metadata.get('Title', 'N/A')}\n"
            f"Resumen: {self.metadata.get('Summary', 'N/A')}\n"
            f"Tipo de Estudio: {self.metadata.get('StudyType', 'N/A')}\n"
            f"Paises donde se desarrolla el estudio: {self.metadata.get('Countries', 'N/A')}\n"
            f"Fase en que se encuentra el estudio: {self.metadata.get('Phases', 'N/A')}\n"
            f"Identificación en ClinicaTrial: {self.metadata.get('IDestudio', 'N/A')}.\n\n"
        )

# Clase para manejar el índice HNSWlib (para búsqueda de similitudes)
class HNSWIndex:
    def __init__(self, embeddings, metadata=None, space='cosine', ef_construction=200, M=16):
        """
        Inicializa un índice HNSWlib con embeddings y metadatos asociados.

        :param embeddings: Matriz de embeddings (vectores de características).
        :param metadata: Metadatos asociados a cada item en el índice (por defecto vacío).
        :param space: Espacio de distancia para el índice (por defecto 'cosine').
        :param ef_construction: Parámetro para la construcción del índice (por defecto 200).
        :param M: Parámetro para el número de conexiones (por defecto 16).
        """
        self.dimension = embeddings.shape[1]  # La dimensión debe coincidir con el tamaño de los embeddings
        self.index = hnswlib.Index(space=space, dim=self.dimension)  # Crear el índice HNSWlib con el espacio dado
        # Inicializar el índice con un número máximo de elementos (igual al número de embeddings)
        self.index.init_index(max_elements=embeddings.shape[0], ef_construction=ef_construction, M=M)
        # Agregar los embeddings al índice
        self.index.add_items(embeddings, np.arange(embeddings.shape[0]))
        # Establecer el parámetro ef para las consultas (cómo de exhaustiva será la búsqueda)
        self.index.set_ef(50)  # Puede ajustarse según las necesidades de rendimiento y precisión
        self.metadata = metadata or []  # Asociar metadatos al índice (vacío si no se proporciona)

    def similarity_search(self, query_vector, k=5):
        """
        Realiza una búsqueda de similitud usando el índice HNSWlib, devolviendo los k documentos más similares.

        :param query_vector: El vector de consulta que se comparará con el índice.
        :param k: El número de resultados más similares a devolver (por defecto 5).
        :return: Lista de tuplas con los metadatos de los documentos más similares y sus distancias.
        """
        # Ejecutar la consulta k-NN (k vecinos más cercanos) sobre el índice
        labels, distances = self.index.knn_query(query_vector, k=k)
        # Devolver los metadatos y las distancias de los k vecinos más cercanos
        return [(self.metadata[i], distances[0][j]) for j, i in enumerate(labels[0])]


### **PASO 7: Procesar los Documentos y Crear un Índice HNSWlib para Cada Conjunto de Documentos**
Se procesan los documentos y se crea un índice HNSWlib para cada conjunto de documentos.

- **Función `desdobla_doc`:**
  - **Propósito:** Procesa cada documento en `data2`, extrae los metadatos relevantes y crea un objeto `Document` para cada entrada. Luego, genera un resumen de cada estudio utilizando la información disponible.
  - **Entradas:**
    - `data2['content']`: Lista de documentos con información sobre estudios clínicos.
  - **Salida:**
    - Una lista de objetos `Document` con el resumen del estudio y sus metadatos.
  
- **Proceso de Embeddings y Creación del Índice HNSWlib:**
  - Los resúmenes de los documentos son convertidos a embeddings utilizando el modelo `SentenceTransformer`.
  - Los embeddings se almacenan en un índice HNSWlib, el cual permite realizar búsquedas eficientes de similitud.
  - **Parámetros del índice HNSWlib:** 
    - La dimensión de los embeddings (dependiente del modelo).
    - El espacio de similitud (por defecto 'cosine').
    - Los parámetros `ef_construction` y `M` son configurados al inicializar el índice.

- **Generación y Almacenamiento de Índices:**
  - Se generan y almacenan los embeddings y los índices para cada conjunto de documentos. Cada conjunto se procesa por separado, y los resultados se guardan en listas de `trozos_archivos` e `index_archivos`.
  
- **Resultado:** Los documentos son procesados, y un índice HNSWlib se crea para cada conjunto de documentos, permitiendo búsquedas de similitudes basadas en embeddings.


In [ ]:
# Función para desdoblar los datos de los ensayos clínicos y crear documentos con embeddings
def desdobla_doc(data2):
    """
    Desdobla la información de un conjunto de datos, creando un resumen y un documento para cada entrada.
    Luego, genera los embeddings de los documentos y crea un índice HNSWlib.
    
    :param data2: Los datos que contienen la información de los ensayos clínicos.
    :return: Una lista de documentos y un índice HNSWlib.
    """
    documents = []  # Lista donde se almacenarán los documentos generados
    summaries = []  # Lista donde se almacenarán los resúmenes generados

    # Iterar sobre cada entrada de los datos (cada ensayo clínico)
    for entry in data2['content']:
        # Obtener metadatos del ensayo clínico
        nctId = entry.get("IDestudio", "")
        briefTitle = entry.get("Title", "")
        summary = entry.get("Summary", "")
        studyType = entry.get("StudyType", "")
        country = entry.get("Countries", "")
        overallStatus = entry.get("OverallStatus", "")
        conditions = entry.get("Conditions", "")
        phases = entry.get("Phasess", "")

        # Crear un resumen conciso del ensayo clínico
        Summary = (
            f"The study titled '{briefTitle}', of type '{studyType}', "
            f"is being conducted to investigate the condition(s) {conditions}. "
            f"This study is briefly summarized as follows: {summary}. "
            f"Currently, the study status is {overallStatus}, and it is taking place in {country}. "
            f"The study is classified under {phases} phase. "
            f"For more information, search {nctId} on ClinicalTrials."
        )

        # Crear los metadatos asociados al resumen
        metadata = {
            "Summary": Summary
        }

        # Añadir el documento con su contenido y metadatos a la lista de documentos
        documents.append(Document(Summary, metadata))
        summaries.append(Summary)

    # Crear embeddings para los documentos utilizando el modelo preentrenado (batch_size de 32 para eficiencia)
    embeddings = model.encode([doc.page_content for doc in documents], batch_size=32, show_progress_bar=True)
    embeddings = np.array(embeddings).astype(np.float32)  # Asegurarse de que los embeddings sean de tipo float32
    
    # Crear un índice HNSWlib para realizar búsquedas eficientes de similitud
    vector_store = HNSWIndex(embeddings, metadata=[doc.metadata for doc in documents])
    
    return documents, vector_store  # Retornar la lista de documentos y el índice creado

# Inicialización de listas para almacenar los trozos y los índices de los archivos
trozos_archivos = []
index_archivos = []

# Iterar sobre los documentos cargados previamente
for i in range(len(documentos)):
    trozos, index = desdobla_doc(documentos[i])  # Crear los documentos y el índice para cada archivo
    trozos_archivos.append(trozos)  # Almacenar los trozos generados
    index_archivos.append(index)  # Almacenar el índice generado

# Mensaje de log para confirmar que se han creado los índices HNSWlib
logging.info("Índices HNSWlib creados para todos los documentos.")


### **PASO 8: Traducir Preguntas y Respuestas**
Traduce preguntas y respuestas entre idiomas según sea necesario utilizando Gemini.

- **Función `traducir`:**
  - **Propósito:** Traduce un texto al idioma destino utilizando el modelo Gemini.
  - **Parámetros:**
    - `texto`: El texto a traducir.
    - `idioma_destino`: El idioma al que se debe traducir el texto (por ejemplo, "inglés").
  - **Proceso:**
    - Se crea un mensaje de sistema y uno de usuario con la solicitud de traducción.
    - Se utiliza el modelo Gemini para realizar la traducción.
    - Se mide el tiempo de traducción y se registra.
  - **Salida:** El texto traducido o, en caso de error, el texto original.

- **Función `generate_embedding`:**
  - **Propósito:** Genera un embedding para un texto utilizando el modelo de embeddings configurado.
  - **Parámetros:**
    - `texto`: El texto para el cual se debe generar el embedding.
  - **Proceso:**
    - Se usa el modelo `SentenceTransformer` para generar el embedding.
    - En caso de error, se devuelve un embedding vacío.
  - **Salida:** Un embedding generado o un embedding vacío en caso de error.

- **Función `obtener_contexto`:**
  - **Propósito:** Recupera los trozos de texto más relevantes para responder a una pregunta.
  - **Parámetros:**
    - `pregunta`: La pregunta realizada por el usuario.
    - `index`: El índice HNSWlib con los embeddings de los documentos.
    - `trozos`: La lista de fragmentos de texto de los documentos.
    - `top_k`: El número de trozos relevantes a recuperar.
  - **Proceso:**
    - La pregunta se traduce al inglés.
    - Se genera un embedding de la pregunta traducida.
    - Se utiliza el índice HNSWlib para recuperar los `top_k` trozos más relevantes.
    - Los trozos relevantes se concatenan para formar el contexto final.
  - **Salida:** El contexto relevante concatenado que se utiliza para generar una respuesta.


In [ ]:
# Función para traducir un texto al idioma destino usando Gemini
def traducir(texto, idioma_destino):
    """
    Traduce el texto al idioma especificado utilizando el modelo Gemini.
    
    :param texto: El texto que se desea traducir.
    :param idioma_destino: El idioma al que se desea traducir el texto (ej. "inglés").
    :return: El texto traducido o el texto original en caso de error.
    """
    start_time = time.time()  # Medir el tiempo que tarda la traducción

    # Preparar los mensajes para la API de Gemini
    mensajes = [
        ChatMessage(role="system", content="Actúa como un traductor."),  # Definir el rol del sistema
        ChatMessage(role="user", content=f"Por favor, traduce este texto al {idioma_destino}: {texto}")  # Traducción solicitada
    ]
    
    try:
        # Realizar la traducción a través de la API de Gemini
        respuesta = gemini_llm.chat(mensajes)
        elapsed_time = time.time() - start_time  # Calcular el tiempo que tardó en traducir
        logging.info(f"Traducción completada: {respuesta.message.content.strip()} en {elapsed_time:.2f} segundos.")
        return respuesta.message.content.strip()  # Devolver la traducción eliminando espacios extra
    except Exception as e:
        # En caso de error, se logea el problema y se devuelve el texto original como fallback
        logging.error(f"Error al traducir: {e}")
        return texto  # Devolver texto original como fallback

# Función para generar embedding de una pregunta usando el modelo de embeddings configurado
def generate_embedding(texto):
    """
    Genera un embedding de texto utilizando el modelo de embeddings preconfigurado.
    
    :param texto: El texto para el cual se generará el embedding.
    :return: El embedding generado para el texto.
    """
    try:
        # Generar el embedding para el texto dado
        embedding = model.encode([texto])
        logging.info(f"Embedding generado para el texto: {texto}")
        return embedding  # Retornar el embedding generado
    except Exception as e:
        # En caso de error, se logea el problema y se retorna un vector de ceros como fallback
        logging.error(f"Error al generar el embedding: {e}")
        return np.zeros((1, 384))  # Retornar un vector de ceros para no interrumpir el flujo, ajustado al tamaño del modelo

# Función para obtener el contexto relevante de la pregunta en función de los documentos y su índice
def obtener_contexto(pregunta, index, trozos, top_k=50):
    """
    Recupera los trozos de texto más relevantes para responder la pregunta, traduciendo la pregunta 
    al inglés antes de buscar en el índice de HNSWlib.
    
    Parámetros:
    - pregunta (str): La pregunta formulada por el usuario.
    - index (HNSWIndex): Índice HNSWlib con embeddings de documentos.
    - trozos (list): Lista de textos troceados relacionados con los documentos.
    - top_k (int): Número de trozos más relevantes a recuperar.

    Retorna:
    - str: El contexto concatenado de los trozos más relevantes.
    """
    try:
        # Traducir la pregunta al inglés para realizar la búsqueda en el índice
        pregunta_en_ingles = traducir(pregunta, "inglés")
        logging.info(f"Pregunta traducida al inglés: {pregunta_en_ingles}")

        # Generar el embedding de la pregunta traducida
        pregunta_emb = generate_embedding(pregunta_en_ingles)
        logging.info("Embedding generado para la pregunta.")

        # Realizar la búsqueda de los trozos más relevantes en el índice HNSWlib
        results = index.similarity_search(pregunta_emb, k=top_k)
        
        # Concatenar los resúmenes de los documentos más relevantes
        texto = ""
        for entry in results:
            resum = entry[0]["Summary"]  # Obtener el resumen de cada documento
            texto += resum + "\n"  # Concatenar todos los resúmenes

        contexto = texto  # El contexto es el conjunto de trozos relevantes
        logging.info("Contexto relevante recuperado para la pregunta.")
        return contexto  # Retornar el contexto recuperado
    except Exception as e:
        # En caso de error, se logea el problema y se retorna un contexto vacío
        logging.error(f"Error al obtener el contexto: {e}")
        return ""  # Retornar un contexto vacío como fallback en caso de error

### **PASO 9: Generar Respuestas**
Clasificar las preguntas en categorías, generar prompts específicos para cada categoría, detectar saludos y generar respuestas detalladas utilizando Gemini.


- **Función `categorizar_pregunta`:**
  - **Propósito:** Clasifica una pregunta en categorías específicas, como "tratamiento", "ensayo clínico", "resultado", y "prevención".
  - **Proceso:**
    - Se define un diccionario con las categorías y las palabras clave asociadas a cada una.
    - La función revisa si alguna palabra clave de cada categoría está presente en la pregunta.
    - Si encuentra una coincidencia, devuelve la categoría correspondiente.
  - **Salida:** La categoría a la que pertenece la pregunta (e.g., "tratamiento", "ensayo", "resultado", "prevención").

- **Función `generar_prompt`:**
  - **Propósito:** Genera un prompt específico para el modelo Gemini según la categoría de la pregunta.
  - **Parámetros:**
    - `categoria`: La categoría de la pregunta (e.g., "tratamiento").
    - `pregunta`: La pregunta del usuario.
  - **Proceso:**
    - La función crea un prompt específico basado en la categoría.
    - Si la categoría no es una de las predefinidas, se devuelve un prompt general sobre ensayos clínicos.
  - **Salida:** Un prompt específico para el modelo de IA.

- **Función `es_saludo`:**
  - **Propósito:** Detecta si la pregunta es un saludo.
  - **Parámetros:**
    - `pregunta`: La pregunta del usuario.
  - **Proceso:**
    - Se comprueba si la pregunta contiene alguna de las palabras clave de saludo (como "hola", "buen día", etc.).
  - **Salida:** `True` si es un saludo, `False` en caso contrario.

- **Función `responder_saludo`:**
  - **Propósito:** Responde con un saludo predefinido.
  - **Proceso:**
    - Si se detecta un saludo, la función responde con una de las respuestas predefinidas.
  - **Salida:** Una respuesta de saludo aleatoria.

- **Función `generar_respuesta`:**
  - **Propósito:** Genera una respuesta completa utilizando el contexto y el prompt específico.
  - **Parámetros:**
    - `pregunta`: La pregunta del usuario.
    - `contexto`: El contexto relevante que se recuperó de los documentos.
    - `prompt_especifico`: El prompt generado para la pregunta.
  - **Proceso:**
    - Se envían los mensajes al modelo Gemini para generar la respuesta.
    - Se mide el tiempo de generación de la respuesta.
    - La respuesta se traduce al español usando la función `traducir`.
  - **Salida:** La respuesta generada y traducida al español.

In [ ]:
# Función para categorizar una pregunta en categorías específicas como 'tratamiento', 'ensayo clínico', etc.
def categorizar_pregunta(pregunta):
    """
    Clasifica la pregunta en categorías específicas como 'tratamiento', 'ensayo clínico', etc.

    Parámetros:
    - pregunta (str): La pregunta del usuario.

    Retorna:
    - str: La categoría a la que pertenece la pregunta (por ejemplo, 'tratamiento', 'ensayo', etc.).
    """
    categorias = {
        "tratamiento": ["tratamiento", "medicación", "cura", "terapia", "fármaco"],  # Palabras clave asociadas al tratamiento
        "ensayo": ["ensayo", "estudio", "prueba", "investigación", "trial"],  # Palabras clave asociadas a ensayos clínicos
        "resultado": ["resultado", "efectividad", "resultados", "éxito", "fracaso"],  # Palabras clave asociadas a resultados
        "prevención": ["prevención", "previene", "evitar", "reducción de riesgo"]  # Palabras clave asociadas a prevención
    }

    # Verificar si alguna de las palabras clave de cada categoría está presente en la pregunta
    for categoria, palabras in categorias.items():
        if any(palabra in pregunta.lower() for palabra in palabras):
            return categoria  # Retornar la categoría si se encuentra una coincidencia
    return "general"  # Si no se encuentra ninguna coincidencia, se devuelve 'general'

# Función para generar un prompt específico basado en la categoría de la pregunta
def generar_prompt(categoria, pregunta):
    """
    Genera un prompt específico para cada categoría de pregunta.

    Parámetros:
    - categoria (str): La categoría de la pregunta (por ejemplo, 'tratamiento', 'ensayo', etc.).
    - pregunta (str): La pregunta del usuario.

    Retorna:
    - str: Un prompt generado según la categoría de la pregunta.
    """
    prompts = {
        "tratamiento": f"Proporciona información detallada sobre tratamientos relacionados con: {pregunta}.",  # Prompt para tratamiento
        "ensayo": f"Describe los ensayos clínicos actuales relacionados con: {pregunta}.",  # Prompt para ensayo clínico
        "resultado": f"Explica los resultados más recientes de ensayos clínicos sobre: {pregunta}.",  # Prompt para resultados
        "prevención": f"Ofrece estrategias de prevención para: {pregunta}."  # Prompt para prevención
    }
    
    # Retornar el prompt asociado a la categoría o un mensaje por defecto si no se encuentra la categoría
    return prompts.get(categoria, "Por favor, responde la pregunta sobre ensayos clínicos.")

# Función para detectar si una pregunta es un saludo
def es_saludo(pregunta):
    """
    Detecta si una pregunta es un saludo.

    Parámetros:
    - pregunta (str): La pregunta del usuario.

    Retorna:
    - bool: True si la pregunta es un saludo, False en caso contrario.
    """
    saludos = ["hola", "buen día", "buenas", "cómo estás", "cómo te llamas", "qué tal", "estás bien", "buenas tardes", "buenas noches"]
    
    # Verificar si alguna de las palabras clave de saludo está presente en la pregunta
    return any(saludo in pregunta.lower() for saludo in saludos)

# Función para generar una respuesta a un saludo
def responder_saludo():
    """
    Responde a un saludo con una de las respuestas predefinidas.

    Retorna:
    - str: Una respuesta aleatoria de saludo.
    """
    saludos_respuestas = [
        "¡Hola! Estoy para ayudarte con información sobre ensayos clínicos. ¿En qué puedo asistirte hoy?",
        "¡Buenas! Tenés alguna pregunta sobre ensayos clínicos en enfermedades neuromusculares?",
        "¡Hola! ¿Cómo puedo ayudarte con tus consultas sobre ensayos clínicos?"
    ]
    
    # Retornar una respuesta de saludo aleatoria
    return random.choice(saludos_respuestas)

# Función para generar una respuesta a una pregunta usando el contexto y un prompt específico
def generar_respuesta(pregunta, contexto, prompt_especifico):
    """
    Genera una respuesta utilizando el contexto proporcionado y un prompt específico.
    
    Parámetros:
    - pregunta (str): La pregunta del usuario.
    - contexto (str): El contexto relacionado a la pregunta.
    - prompt_especifico (str): El prompt específico que se utiliza para generar la respuesta.
    
    Retorna:
    - str: La respuesta generada (traducida al español).
    """
    start_time = time.time()  # Medir el tiempo de generación de respuesta

    # Preparar los mensajes para la API de Gemini, incluyendo el prompt y el contexto
    mensajes = [
        ChatMessage(role="system", content="Eres un experto médico."),  # Rol del sistema como experto médico
        ChatMessage(role="user", content=f"{prompt_especifico}\nContexto: {contexto}\nPregunta: {pregunta}")  # Incluir la pregunta y el contexto
    ]
    
    try:
        # Realizar la consulta al modelo Gemini para obtener la respuesta
        respuesta = gemini_llm.chat(mensajes)
        elapsed_time = time.time() - start_time  # Calcular el tiempo de generación de respuesta
        logging.info(f"Respuesta generada en inglés: {respuesta.message.content.strip()} en {elapsed_time:.2f} segundos.")

        # Traducir la respuesta generada al español
        respuesta_en_espanol = traducir(respuesta.message.content, "español")
        logging.info(f"Respuesta traducida al español: {respuesta_en_espanol}")
        return respuesta_en_espanol  # Retornar la respuesta traducida
    except Exception as e:
        # En caso de error, se logea el problema y se retorna un mensaje de error
        logging.error(f"Error al generar la respuesta: {e}")
        return "Lo siento, ocurrió un error al generar la respuesta."  # Mensaje de error en caso de fallo


### **PASO 10: Función Principal para Responder Preguntas**
Integra todos los procesos anteriores en una función principal que maneja la categorización, traducción, recuperación de contexto y generación de respuestas, con optimización mediante un sistema de caché.

- **Función `generar_hash`:**
  - **Propósito:** Genera un hash único (SHA-256) para una pregunta dada. Este hash se utiliza para crear un identificador único para cada pregunta y facilitar el almacenamiento en caché.
  - **Entrada:** Pregunta (texto).
  - **Salida:** Hash único de la pregunta.

- **Función `obtener_respuesta_cacheada`:**
  - **Propósito:** Recupera la respuesta de la pregunta desde un archivo de caché si ya se ha respondido previamente.
  - **Entrada:** Pregunta (texto).
  - **Salida:** Respuesta cacheada (si existe), de lo contrario `None`.

- **Función `guardar_respuesta_cacheada`:**
  - **Propósito:** Guarda una respuesta generada en un archivo de caché para optimizar futuras consultas.
  - **Entrada:** Pregunta (texto) y Respuesta generada.
  - **Salida:** Ninguna, pero guarda la respuesta en el sistema de caché.

- **Función `responder_pregunta`:**
  - **Propósito:** Función principal que integra todos los pasos anteriores:
    - **Categoriza la pregunta** para identificar la temática.
    - **Genera un prompt específico** para la pregunta basada en su categoría.
    - **Recupera el contexto relevante** de la base de datos usando el índice HNSWlib.
    - **Genera una respuesta** utilizando el modelo Gemini y el contexto recuperado.
    - **Gestiona el caché** para optimizar las respuestas futuras.
  - **Entrada:**
    - `pregunta`: La pregunta que el usuario hace.
    - `index`: El índice HNSWlib para realizar búsquedas.
    - `trozos`: Los documentos o fragmentos que contienen información relevante.
  - **Proceso:**
    1. Revisa si la respuesta ya está en caché.
    2. Si no, categoriza la pregunta y genera el prompt específico.
    3. Recupera el contexto adecuado para la pregunta.
    4. Genera la respuesta utilizando el modelo y el contexto.
    5. Guarda la respuesta en el caché.
  - **Salida:** Respuesta generada para la pregunta del usuario.

In [ ]:
# Función para generar un hash SHA-256 de la pregunta
def generar_hash(pregunta):
    """
    Genera un hash SHA-256 para una pregunta dada.

    Parámetros:
    - pregunta (str): La pregunta del usuario.

    Retorna:
    - str: El hash SHA-256 generado de la pregunta.
    """
    # Se utiliza SHA-256 para generar un hash único basado en la pregunta
    return hashlib.sha256(pregunta.encode('utf-8')).hexdigest()

# Función para obtener la respuesta cacheada de una pregunta si ya existe
def obtener_respuesta_cacheada(pregunta):
    """
    Obtiene la respuesta de un archivo de caché si ya existe.

    Parámetros:
    - pregunta (str): La pregunta del usuario.

    Retorna:
    - str: La respuesta cacheada, si existe.
    - None: Si no existe una respuesta cacheada.
    """
    hash_pregunta = generar_hash(pregunta)  # Generar el hash para identificar el archivo cacheado
    archivo_cache = f"cache/{hash_pregunta}.json"  # Nombre del archivo de caché basado en el hash
    if os.path.exists(archivo_cache):  # Verificar si el archivo cacheado existe
        try:
            with open(archivo_cache, "r", encoding='utf-8') as f:
                datos = json.load(f)  # Leer el archivo JSON
                return datos.get("respuesta", None)  # Retornar la respuesta cacheada
        except Exception as e:
            logging.error(f"Error al leer el caché para la pregunta '{pregunta}': {e}")  # Log de error si algo falla
            return None
    return None  # Si no existe el archivo, retornar None

# Función para guardar la respuesta en caché para futuras consultas
def guardar_respuesta_cacheada(pregunta, respuesta):
    """
    Guarda la respuesta en caché para futuras consultas.

    Parámetros:
    - pregunta (str): La pregunta del usuario.
    - respuesta (str): La respuesta generada para la pregunta.
    """
    hash_pregunta = generar_hash(pregunta)  # Generar el hash de la pregunta para identificar el archivo cacheado
    archivo_cache = f"cache/{hash_pregunta}.json"  # Nombre del archivo cacheado
    try:
        os.makedirs(os.path.dirname(archivo_cache), exist_ok=True)  # Asegura que el directorio exista
        with open(archivo_cache, "w", encoding='utf-8') as f:
            # Guardar la pregunta y respuesta en formato JSON en el archivo cacheado
            json.dump({"pregunta": pregunta, "respuesta": respuesta}, f, ensure_ascii=False, indent=4)
        logging.info(f"Respuesta cacheada para la pregunta: '{pregunta}'")  # Log cuando se guarda la respuesta
    except Exception as e:
        logging.error(f"Error al guardar la respuesta en caché para la pregunta '{pregunta}': {e}")  # Log de error si algo falla

# Función principal para responder una pregunta, integrando todos los pasos
def responder_pregunta(pregunta, index, trozos):
    """
    Integra todos los pasos: categorización, traducción, recuperación de contexto y generación de respuestas.
    Incluye manejo de caché para optimizar la velocidad de respuesta.

    Parámetros:
    - pregunta (str): La pregunta del usuario.
    - index: El índice de búsqueda para obtener contexto relevante.
    - trozos: Los fragmentos de datos de los que se puede recuperar información.

    Retorna:
    - str: La respuesta generada para la pregunta.
    """
    try:
        # Verificar si la respuesta ya está en el caché
        respuesta_cacheada = obtener_respuesta_cacheada(pregunta)
        if respuesta_cacheada:
            logging.info(f"Respuesta obtenida del caché para la pregunta: '{pregunta}'")  # Log cuando se usa el caché
            return respuesta_cacheada  # Retornar la respuesta cacheada si existe
        
        # Categorizar la pregunta para determinar el tipo de información necesaria
        categoria = categorizar_pregunta(pregunta)
        logging.info(f"Categoría de la pregunta: {categoria}")  # Log de la categoría de la pregunta

        # Generar un prompt específico basado en la categoría de la pregunta
        prompt_especifico = generar_prompt(categoria, pregunta)  # Generar el prompt según la categoría
        logging.info(f"Prompt específico generado: {prompt_especifico}")  # Log del prompt generado

        # Obtener el contexto relevante para la pregunta
        contexto = obtener_contexto(pregunta, index, trozos)
        if not contexto:  # Si no se encontró contexto, manejar el caso de forma adecuada
            logging.warning("No se encontró un contexto relevante para la pregunta.")  # Log si no hay contexto
            respuesta = "No pude encontrar información relevante para responder tu pregunta."
            guardar_respuesta_cacheada(pregunta, respuesta)  # Guardar una respuesta por defecto en el caché
            return respuesta

        # Generar la respuesta usando el contexto y el prompt específico
        respuesta = generar_respuesta(pregunta, contexto, prompt_especifico)
        
        # Guardar la respuesta generada en el caché para futuras consultas
        guardar_respuesta_cacheada(pregunta, respuesta)
        
        return respuesta  # Retornar la respuesta generada
    except Exception as e:
        logging.error(f"Error en el proceso de responder pregunta: {e}")  # Log de error si algo falla
        return "Ocurrió un error al procesar tu pregunta."  # Mensaje de error si ocurre un fallo

### **PASO 11: Interfaz CLI**
Implementar una interfaz de línea de comandos (CLI) que permita a los usuarios interactuar con el chatbot de manera directa.

In [ ]:
# Asegurarse de que el directorio de caché exista al inicio
if __name__ == "__main__":
    # Crear el directorio "cache" si no existe
    os.makedirs("cache", exist_ok=True)  # El parámetro exist_ok=True evita un error si el directorio ya existe

    print("Bienvenido al chatbot de ensayos clínicos")
    print("Conversemos sobre Ensayos Clínicos\n de las siguientes enfermedades neuromusculares: Distrofia Muscular de Duchenne o de Becker, Enfermedad de Pompe, Distrofia Miotónica o Enfermedad de almacenamiento de glucosa")
    print("Escribe tu pregunta, dejando claramente expresada la enfermedad sobre la que quieres conocer información de ensayos médicos. Escribe 'salir' para terminar.")
    
    while True:
        # Solicitar la pregunta al usuario
        pregunta = input("Tu pregunta: ").strip()  # Se elimina cualquier espacio en blanco antes o después de la pregunta

        # Verificar si el usuario desea salir
        if pregunta.lower() in ['salir', 'chau', 'exit', 'quit']:
            print("¡Chau!")
            logging.info("El usuario ha finalizado la sesión.")  # Registrar en el log que el usuario ha terminado la sesión
            break  # Salir del ciclo si el usuario escribe "salir", "chau", "exit" o "quit"

        # Verificar si la pregunta es un saludo
        if es_saludo(pregunta):
            # Si la pregunta es un saludo, responder con un mensaje de saludo
            respuesta_saludo = responder_saludo()
            print(respuesta_saludo)
            logging.info("Se detectó un saludo del usuario.")  # Registrar en el log que se detectó un saludo
            continue  # Volver al inicio del ciclo para permitir otra pregunta

        # Identificar la enfermedad relacionada con la pregunta
        idn = doc_enfermedad(pregunta)  # Llamar a la función que determina la enfermedad mencionada en la pregunta
        index = index_archivos[idn]  # Obtener el índice relacionado con la enfermedad
        trozos = trozos_archivos[idn]  # Obtener los trozos de información de la enfermedad

        # Generar la respuesta para la pregunta del usuario utilizando el index y los trozos correspondientes
        respuesta = responder_pregunta(pregunta, index, trozos)  # Responder basándose en el contexto recuperado
        print(f"Respuesta: {respuesta}")  # Imprimir la respuesta generada

**Flujo del CLI:**
 - **Entrada de usuario:** El usuario puede ingresar una pregunta.
 - **Chequeo de saludo:** Si la pregunta es un saludo, se responde adecuadamente.
 - **Detección de enfermedad:** Si la pregunta está relacionada con una enfermedad específica, el chatbot utiliza el índice correspondiente.
 - **Generación de respuesta:** El chatbot genera y devuelve una respuesta adecuada basada en la pregunta y el contexto.
 - **Cierre de sesión:** Si el usuario escribe 'salir' o palabras relacionadas, el chatbot termina la interacción.